In [1]:
import pandas as pd
import numpy as np
import re

# 1. 데이터 불러오기

In [2]:
df = pd.read_csv("./data/저출산 소개팅_설문조사_시나리오추가.csv")

In [3]:
df.head()

,타임스탬프,0. 당신의 성함,0. 당신의 이상형,1-1. 희망하는 자녀 수,1-2. 희망하는 자녀 구성,1-3. 자녀 갖고 싶은 시기,1-4. 생물학적 출산이 어려움 발생 시 대안,"""1. 자녀 계획 및 가족 구성 항목""에 대해 중요도","아이가 ""오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?",평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.,...,맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?,"양가 어르신들이 본인의 가치관과 다른 육아 조언(예: ""애를 너무 손타게 키운다"", ""사탕 좀 주면 어떠냐"")을 하실 때 당신의 생각은?","주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?",아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?,4-1. 자녀 교육비/양육비 지출 비중,"4-2. 육아 휴직, 양육 부담","""4. 경제적 지원 및 가사 분담""에 대해 중요도","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?","""5. 자녀 가치관""에 대해 중요도",Unnamed: 23
0,2026/02/27 3:53:18 PM GMT+9,강현준,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,4,...,2,4,5,4,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3,https://drive.google.com/u/0/open?usp=forms_we...
1,2026/02/27 4:15:15 PM GMT+9,임경빈,꼬부기상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,4,...,4,4,4,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",3,NaN
2,2026/02/27 4:19:32 PM GMT+9,이정미,곰상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,5,...,4,2,4,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",3,"회복탄력성, 생활력 강한 사람",4,NaN
3,2026/02/27 4:29:59 PM GMT+9,쳌,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,4,...,4,2,5,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"자신이 좋아하는 일, 행복",2,NaN
4,2026/02/27 4:30:18 PM GMT+9,김용희,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,4,...,5,2,2,3,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",2,"자신이 좋아하는 일, 행복",3,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 24 columns):
 #   Column                                                                                 Non-Null Count  Dtype
---  ------                                                                                 --------------  -----
 0   타임스탬프                                                                                  34 non-null     str  
 1   0. 당신의 성함                                                                              34 non-null     str  
 2   0. 당신의 이상형                                                                             34 non-null     str  
 3   1-1. 희망하는 자녀 수                                                                         34 non-null     str  
 4   1-2. 희망하는 자녀 구성                                                                        34 non-null     str  
 5   1-3. 자녀 갖고 싶은 시기                                                                       34 non-null     st

# 2. 데이터 전처리

In [5]:
df_raw = df.copy()

## (1) 컬럼 정리

### 컬럼 rename

In [7]:
# 1) 컬럼명 정규화 함수 (공백/따옴표/줄바꿈 정리)
def clean_colname(col: str) -> str:
    col = str(col).strip()  # 앞뒤 공백 제거
    col = col.replace("\n", " ").replace("\t", " ") # 줄바꿈/탭 등을 공백으로 변경 
    col = re.sub(r"\s+", " ", col)      # 공백 여러 개 -> 하나
    col = col.replace('"', "")          # 큰따옴표 제거
    return col

In [8]:
# 2) 정규화 적용 + 중복 컬럼명 방지(매우 중요)
cleaned_cols = [clean_colname(c) for c in df.columns]

In [9]:
# 정규화 후 같은 이름이 중복되면 모델링/전처리에서 에러가 나므로, 자동으로 _1, _2 같은 접미사를 붙임
def make_unique(names):
    seen = {}
    out = []
    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}_{seen[n]}")
    return out

df.columns = make_unique(cleaned_cols)

In [10]:
# 3) Unnamed 같은 잔재 컬럼 제거
unnamed_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)

In [11]:

# 4) 짧은 컬럼명으로 rename
rename_map = {
    "타임스탬프": "timestamp",
    "0. 당신의 성함": "user_name",
    "0. 당신의 이상형": "ideal_type",

    "1-1. 희망하는 자녀 수": "p_children_count",
    "1-2. 희망하는 자녀 구성": "p_children_composition",
    "1-3. 자녀 갖고 싶은 시기": "p_children_timing",
    "1-4. 생물학적 출산이 어려움 발생 시 대안": "p_infertility_alternative",

    # 따옴표 제거(clean_colname) 후의 이름을 기준으로 매칭됨
    "1. 자녀 계획 및 가족 구성 항목에 대해 중요도": "imp_family_plan",

    # 시나리오(정수형) 문항들
    '아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?'
    : "sc_toothbrushing",
    "평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다."
    : "sc_bedtime_story",
    "경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?"
    : "sc_competition_2nd",
    "재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?"
    : "sc_talent_education",
    "두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?"
    : "sc_discipline_conflict",
    "한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다."
    : "sc_play_vs_chores",
    "맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?"
    : "sc_grandparents_help",
    # 따옴표는 clean_colname에서 제거되므로, 예시 문장 안 따옴표도 없는 버전으로 매칭
    "양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?"
    : "sc_inlaws_advice",
    "주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?"
    : "sc_rainy_zoo",
    "아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?"
    : "sc_education_fund_risk",

    "4-1. 자녀 교육비/양육비 지출 비중": "e_childcare_cost_share",
    "4-2. 육아 휴직, 양육 부담": "e_parental_leave_burden",
    "4. 경제적 지원 및 가사 분담에 대해 중요도": "imp_econ_housework",

    "5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?": "child_values_open",
    "5. 자녀 가치관에 대해 중요도": "imp_child_values",
}

In [12]:
# rename 시도하기 전에 "매칭이 안 되는 컬럼"이 있는지 점검
missing_in_df = [k for k in rename_map.keys() if k not in df.columns]
if missing_in_df:
    for m in missing_in_df:
        print(" -", m)

In [13]:
# 실제 rename 적용 (존재하는 것만 rename)
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

In [14]:
# 5) 타입 정리 (최소한으로만).
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

C:\Users\user\AppData\Local\Temp\ipykernel_39732\4096760953.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")


In [15]:
# 이름(name)을 index로 설정 + timestamp 삭제

# (1) name 중복 여부 체크
if "user_name" not in df.columns:
    raise ValueError("❌ 'name' 컬럼이 없습니다. rename_map에서 '0. 당신의 성함' 매핑이 제대로 됐는지 확인하세요.")

dup_cnt = df["user_name"].duplicated().sum()
print("🔎 user_name 중복 개수:", dup_cnt)

# (2) 중복이 있다면 인덱스가 깨지므로, 임시로 뒤에 _1, _2 붙여서 유니크하게 만드는 방식
#     (중복이 없으면 그대로 진행)
if dup_cnt > 0:
    df["user_name"] = df["user_name"].astype(str)
    df["user_name"] = df["user_name"] + "_" + df.groupby("user_name").cumcount().astype(str)

# (3) 인덱스 설정
df = df.set_index("user_name").copy()

# (4) timestamp 삭제
if "timestamp" in df.columns:
    df = df.drop(columns=["timestamp"]).copy()

print("✅ index 설정 + timestamp 삭제 완료:", df.shape)
print(df.head(2))
print(df.info())

🔎 user_name 중복 개수: 0
✅ index 설정 + timestamp 삭제 완료: (34, 21)
          ideal_type p_children_count p_children_composition  \
user_name                                                      
강현준             꽃돼지상               2명            딸 1명, 아들 1명   
임경빈             꼬부기상               2명            딸 1명, 아들 1명   

          p_children_timing p_infertility_alternative  imp_family_plan  \
user_name                                                                
강현준                   결혼 즉시        의학적 도움 적극 시도;입양 고려                5   
임경빈            결혼 후 1~2년 이내              의학적 도움 적극 시도                4   

           sc_toothbrushing  sc_bedtime_story  sc_competition_2nd  \
user_name                                                           
강현준                       2                 4                   5   
임경빈                       3                 4                   4   

           sc_talent_education  ...  sc_play_vs_chores  sc_grandparents_help  \
user_name                    

In [16]:
print("\n==== 최종 df 요약 ====")
print("shape:", df.shape)
print("dtypes:\n", df.dtypes)


==== 최종 df 요약 ====
shape: (34, 21)
dtypes:
 ideal_type                     str
p_children_count               str
p_children_composition         str
p_children_timing              str
p_infertility_alternative      str
imp_family_plan              int64
sc_toothbrushing             int64
sc_bedtime_story             int64
sc_competition_2nd           int64
sc_talent_education          int64
sc_discipline_conflict       int64
sc_play_vs_chores            int64
sc_grandparents_help         int64
sc_inlaws_advice             int64
sc_rainy_zoo                 int64
sc_education_fund_risk       int64
e_childcare_cost_share         str
e_parental_leave_burden        str
imp_econ_housework           int64
child_values_open              str
imp_child_values             int64
dtype: object


### 컬럼 분류

In [17]:
# =========================
# 4) 컬럼 분류(그룹 리스트) 만들기
# =========================
# 분류 기준:
# - 자녀 계획 및 가족구성
# - 시나리오기반 자녀가치관 분석
# - 경제적 지원 및 가사분담
# - 자녀 가치관
# + 서술형(기타) 따로
# + 중요도(가중치) 따로

# (A) 중요도/가중치 컬럼
IMPORTANCE_COLS = [
    "imp_family_plan",
    "imp_econ_housework",
    "imp_child_values",
]

# (B) 서술형(기타) 컬럼
TEXT_COLS = [
    "p_children_composition",  # 1-2
    "child_values_open",                  # 5-1
]

# (C) 자녀 계획 및 가족 구성(서술형 제외한 선택형 중심)
PLAN_COLS = [
    "p_children_count",
    "p_children_timing",
    "p_infertility_alternative",
]

# (D) 시나리오 기반(정수형 1~5 같은 응답)
SCENARIO_COLS = [
    "sc_toothbrushing",
    "sc_bedtime_story",
    "sc_competition_2nd",
    "sc_talent_education",
    "sc_discipline_conflict",
    "sc_play_vs_chores",
    "sc_grandparents_help",
    "sc_inlaws_advice",
    "sc_rainy_zoo",
    "sc_education_fund_risk",
]

# (E) 경제적 지원 및 가사 분담
ECON_COLS = [
    "e_childcare_cost_share",
    "e_parental_leave_burden",
]

# (F) 메타(추천 피처에 넣을지 말지는 다음 단계에서 결정)
# - 이상형은 매칭 시스템에서 활용할 여지가 있지만,
#   지금 단계에서는 우선 메타로 따로 빼놓는 게 안전
META_COLS = [
    "ideal_type",
]

# ---- 존재 여부 검증(오타/누락 방지) ----
all_groups = {
    "META_COLS": META_COLS,
    "PLAN_COLS": PLAN_COLS,
    "SCENARIO_COLS": SCENARIO_COLS,
    "ECON_COLS": ECON_COLS,
    "TEXT_COLS": TEXT_COLS,
    "IMPORTANCE_COLS": IMPORTANCE_COLS,
}


In [18]:
# 확인
for gname, cols in all_groups.items():
    not_found = [c for c in cols if c not in df.columns]
    if not_found:
        print(f"❌ {gname} 에 df에 없는 컬럼이 있습니다:", not_found)
    else:
        print(f"✅ {gname} OK ({len(cols)}개)")

# ---- 그룹별로 값 예시 확인 ----
print("\n===== 그룹별 샘플(상위 2행) =====")
for gname, cols in all_groups.items():
    existing = [c for c in cols if c in df.columns]
    print(f"\n[{gname}]")
    display(df[existing].head(2))

✅ META_COLS OK (1개)
✅ PLAN_COLS OK (3개)
✅ SCENARIO_COLS OK (10개)
✅ ECON_COLS OK (2개)
✅ TEXT_COLS OK (2개)
✅ IMPORTANCE_COLS OK (3개)

===== 그룹별 샘플(상위 2행) =====

[META_COLS]


,ideal_type
user_name,
강현준,꽃돼지상
임경빈,꼬부기상



[PLAN_COLS]


,p_children_count,p_children_timing,p_infertility_alternative
user_name,,,
강현준,2명,결혼 즉시,의학적 도움 적극 시도;입양 고려
임경빈,2명,결혼 후 1~2년 이내,의학적 도움 적극 시도



[SCENARIO_COLS]


...

,e_childcare_cost_share,e_parental_leave_burden
user_name,,
강현준,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아"
임경빈,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아"



[TEXT_COLS]


,p_children_composition,child_values_open
user_name,,
강현준,"딸 1명, 아들 1명","자신이 좋아하는 일, 행복"
임경빈,"딸 1명, 아들 1명","회복탄력성, 생활력 강한 사람"



[IMPORTANCE_COLS]


,imp_family_plan,imp_econ_housework,imp_child_values
user_name,,,
강현준,5,1,3
임경빈,4,2,3


# 3. 스케일링

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
TEXT_COLS

['p_children_composition', 'child_values_open']

In [22]:
df["p_children_composition"].value_counts()

p_children_composition
딸 1명, 아들 1명    15
오직 딸           15
되는대로            1
딸2 아들1          1
아들1 딸2          1
딸 2명, 아들 1명     1
Name: count, dtype: int64

In [23]:
df["child_values_open"].value_counts()

child_values_open
자신이 좋아하는 일, 행복                                                                                                      14
회복탄력성, 생활력 강한 사람                                                                                                    10
도덕적, 타인 배려                                                                                                           8
경제적 성공, 사회적 지위                                                                                                       1
자녀의 입에서 나는 엄마, 아빠처럼 살래요 라는 말과 목표가 나오도록 / 가장 존경하는 사람이 누구인가 라고 자녀에게 누군가 물어보면, 우리 엄마 아빠요 라고 대답할 수 있는 사람이 되었으면 좋겠습니다     1
Name: count, dtype: int64

In [24]:
# all_categories = {
#    'ideal_type': ['dog', 'cat', 'deer', 'rabbit', 'kkobugi','bear','quokka','mouse','salamander','fox','horse','racoon','pig','hamster','wolf','koala','duck','monkey','dino','alpaca'],
#    'p_children_count': ['one', 'two', 'three', 'more'],
#    'p_children_composition': ['only_dau', 'only_son', 'one_dau_one_son'],
#    'p_children_timing': ['immediately', 'within_1_2_years', 'within_3_5_years', 'after_stability'],
#    'p_infertility_alternative': ['try_medical_help', 'adoption', 'childfree'],
#    'e_childcare_cost_share': ['child_education', 'retirement'],
#    'e_parental_leave_burden': ['higher_work_other_care', 'dual_income'],
#    'child_values_open': ['economic_success', 'morality', 'happiness', 'resilience_life']
# }

## (1) 인코딩

In [25]:
# 원핫인코딩 컬럼명 설정
all_categories = {
    'ideal_type': ['강아지상', '고양이상', '사슴상', '토끼상', '꼬부기상','곰상','쿼카상','쥐상','도룡뇽상','여우상','말상','너구리상','꽃돼지상','햄스터상','늑대상','코알라상','오리상','원숭이','공룡상','알파카상, 라마상'],
    'p_children_count': ['1명', '2명', '3명', '그 이상'],
    'p_children_composition': ['오직 딸', '오직 아들', '딸 1명, 아들 1명'],
    'p_children_timing': ['결혼 즉시', '결혼 후 1~2년 이내', '결혼 후 3~5년 이내', '경제적 안정 후'],
    'p_infertility_alternative': ['의학적 도움 적극 시도', '입양 고려', '무자녀'],
    'e_childcare_cost_share': ['노후보단 자녀 교육', '노후 먼저, 남는 예산으로 지원'],
    'e_parental_leave_burden': ['경제력 높은 사람 일하고, 한명은 전담 육아', '맞벌이하면서 외부 도움(조부모, 시터)'],
    'child_values_open': ['경제적 성공, 사회적 지위', '도덕적, 타인 배려', '자신이 좋아하는 일, 행복', '회복탄력성, 생활력 강한 사람']
}

In [26]:
target_cols = [
    'ideal_type', 
    'p_children_count', 
    'p_children_composition', 
    'p_children_timing', 
    'p_infertility_alternative', 
    'e_childcare_cost_share', 
    'e_parental_leave_burden', 
    'child_values_open'
]

In [27]:
from pandas.api.types import CategoricalDtype

def encode_with_schema_v3(df, categories_dict):
    result_df = df.copy()
    
    for col, categories in categories_dict.items():
        if col not in result_df.columns:
            continue
            
        # 1. '기타' 처리 (단일 선택형)
        if col != 'p_infertility_alternative':
            # 카테고리 리스트에 '기타'가 없다면 추가
            current_categories = categories.copy()
            if '기타' not in current_categories:
                current_categories.append('기타')
            
            # [수정 포인트] 리스트에 없는 값은 모두 '기타'라는 문자열로 변경
            result_df[col] = result_df[col].apply(lambda x: x if x in categories else '기타')
            
            # 카테고리 타입 적용
            dtype = CategoricalDtype(categories=current_categories)
            result_df[col] = result_df[col].astype(dtype)
            
            # 원-핫 인코딩
            dummies = pd.get_dummies(result_df[col], prefix=col)
            result_df = pd.concat([result_df, dummies], axis=1)
            result_df.drop(columns=[col], inplace=True)
            
        else:
            # 2. 다중 선택형(1-4번) 처리 (기존과 동일하게 유지하되 확실히 체크)
            for cat in categories:
                result_df[f"{col}_{cat}"] = result_df[col].astype(str).apply(
                    lambda x: 1 if cat in x else 0
                )
            
            # 정의되지 않은 답변이 하나라도 섞여 있다면 '기타'로 표시
            result_df[f"{col}_기타"] = result_df[col].apply(
                lambda x: 1 if any(item.strip() not in categories for item in str(x).split(';')) else 0
            )
            result_df.drop(columns=[col], inplace=True)
            
    return result_df

# 실행
df_encoded = encode_with_schema_v3(df, all_categories)

# 확인: '1-2. 희망하는 자녀 구성_기타' 열이 있는지 확인

In [28]:
target_check = [c for c in df_encoded.columns if 'p_children_composition' in c]
print("생성된 컬럼들:", target_check)

생성된 컬럼들: ['p_children_composition_오직 딸', 'p_children_composition_오직 아들', 'p_children_composition_딸 1명, 아들 1명', 'p_children_composition_기타']


In [29]:
df_encoded.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 강현준 to 유현희
Data columns (total 63 columns):
 #   Column                                            Non-Null Count  Dtype
---  ------                                            --------------  -----
 0   imp_family_plan                                   34 non-null     int64
 1   sc_toothbrushing                                  34 non-null     int64
 2   sc_bedtime_story                                  34 non-null     int64
 3   sc_competition_2nd                                34 non-null     int64
 4   sc_talent_education                               34 non-null     int64
 5   sc_discipline_conflict                            34 non-null     int64
 6   sc_play_vs_chores                                 34 non-null     int64
 7   sc_grandparents_help                              34 non-null     int64
 8   sc_inlaws_advice                                  34 non-null     int64
 9   sc_rainy_zoo                                      34 non-n

### 가중치 설정

In [30]:
#가중치 설정 (중요도 설문 조사 값을 통해 각 섹션의 값 가중치 조절)

import numpy as np

def apply_weighted_logic(df):
    #섹션별 질문 컬럼 분류 

    w1_col = 'imp_family_plan'
    w4_col = 'imp_econ_housework'
    w5_col = 'imp_child_values'

    section_1_cols = [c for c in df.columns if c.startswith('p_')]
    section_4_cols = [c for c in df.columns if c.startswith('e_')]
    section_5_cols = [c for c in df.columns if c.startswith('child_')]

    # 중요도 가중치 계산 (1~5점을 0.2~1.0 가중치로 변환)
    w1 = (df[w1_col].values.astype(float) / 5.0).reshape(-1, 1)
    w4 = (df[w4_col].values.astype(float) / 5.0).reshape(-1, 1)
    w5 = (df[w5_col].values.astype(float) / 5.0).reshape(-1, 1)

    # 각 행별로 섹션 데이터에 가중치 곱하기
    df[section_1_cols] = df[section_1_cols].values * w1
    df[section_4_cols] = df[section_4_cols].values * w4
    df[section_5_cols] = df[section_5_cols].values * w5

    #중요도 칼럼은 제외시키기 
    df_weighted = df.drop(columns=[w1_col, w4_col, w5_col])

    return df_weighted

df_weighted = apply_weighted_logic(df_encoded)

In [31]:
# 시나리오 질문 기반 파생 변수 생성

def create_upbringing_traits(df):
    # 1. 섹션별 질문 매핑 (질문 문구의 핵심 키워드로 매칭)
    trait_mapping = {
        'parenting_style_SF': ['sc_toothbrushing','sc_bedtime_story'],
        'education_priority_AH': ['sc_competition_2nd','sc_talent_education'],
        'co_parenting_mode_EL': ['sc_discipline_conflict','sc_play_vs_chores'],
        'family_boundary_BT': ['sc_grandparents_help','sc_inlaws_advice'],
        'planning_risk_PG': ['sc_rainy_zoo','sc_education_fund_risk']
    }

    # 2. 각 섹션별 합산 점수 계산
    for trait_name, questions in trait_mapping.items():
        # 질문이 데이터에 있는지 확인 후 합산 (1~5점 질문 2개이므로 합은 2~10점)
        available_cols = [q for q in questions if q in df.columns]
        if len(available_cols) == 2:
            df[trait_name] = df[available_cols[0]].astype(int) + df[available_cols[1]].astype(int)
        else:
            print(f"경고: {trait_name}를 위한 질문 컬럼을 찾을 수 없습니다.")

    # 3. 기존 개별 시나리오 질문 컬럼은 삭제 (이제 '성향 점수'가 그 역할을 대신함)
    all_scenario_cols = [q for sublist in trait_mapping.values() for q in sublist if q in df.columns]
    df_traits = df.drop(columns=all_scenario_cols)
    
    return df_traits

In [32]:
# 실행
df_with_traits = create_upbringing_traits(df_weighted)

In [33]:
# 결과 확인
print("새로 생성된 성향 컬럼들:")
print(df_with_traits[['parenting_style_SF', 'education_priority_AH', 'co_parenting_mode_EL', 'family_boundary_BT', 'planning_risk_PG']].head())

새로 생성된 성향 컬럼들:
           parenting_style_SF  education_priority_AH  co_parenting_mode_EL  \
user_name                                                                    
강현준                         6                      9                     8   
임경빈                         7                      6                     8   
이정미                         9                      9                     9   
쳌                           7                      9                     8   
김용희                         5                      6                     5   

           family_boundary_BT  planning_risk_PG  
user_name                                        
강현준                         6                 9  
임경빈                         8                 7  
이정미                         6                 6  
쳌                           6                 7  
김용희                         7                 5  


In [34]:
df_with_traits.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 강현준 to 유현희
Data columns (total 55 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   ideal_type_강아지상                                   34 non-null     bool   
 1   ideal_type_고양이상                                   34 non-null     bool   
 2   ideal_type_사슴상                                    34 non-null     bool   
 3   ideal_type_토끼상                                    34 non-null     bool   
 4   ideal_type_꼬부기상                                   34 non-null     bool   
 5   ideal_type_곰상                                     34 non-null     bool   
 6   ideal_type_쿼카상                                    34 non-null     bool   
 7   ideal_type_쥐상                                     34 non-null     bool   
 8   ideal_type_도룡뇽상                                   34 non-null     bool   
 9   ideal_type_여우상                      

In [35]:
df_with_traits.head()

,ideal_type_강아지상,ideal_type_고양이상,ideal_type_사슴상,ideal_type_토끼상,ideal_type_꼬부기상,ideal_type_곰상,ideal_type_쿼카상,ideal_type_쥐상,ideal_type_도룡뇽상,ideal_type_여우상,...,"child_values_open_경제적 성공, 사회적 지위","child_values_open_도덕적, 타인 배려","child_values_open_자신이 좋아하는 일, 행복","child_values_open_회복탄력성, 생활력 강한 사람",child_values_open_기타,parenting_style_SF,education_priority_AH,co_parenting_mode_EL,family_boundary_BT,planning_risk_PG
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,False,False,False,False,False,False,False,False,False,False,...,0.0,0.0,0.6,0.0,0.0,6,9,8,6,9
임경빈,False,False,False,False,True,False,False,False,False,False,...,0.0,0.0,0.0,0.6,0.0,7,6,8,8,7
이정미,False,False,False,False,False,True,False,False,False,False,...,0.0,0.0,0.0,0.8,0.0,9,9,9,6,6
쳌,False,True,False,False,False,False,False,False,False,False,...,0.0,0.0,0.4,0.0,0.0,7,9,8,6,7
김용희,False,False,False,False,False,False,False,False,False,True,...,0.0,0.0,0.6,0.0,0.0,5,6,5,7,5


In [36]:
def scale_traits_fixed(df):
    trait_cols = ['parenting_style_SF', 'education_priority_AH', 'co_parenting_mode_EL', 'family_boundary_BT', 'planning_risk_PG']
    
    for col in trait_cols:
        if col in df.columns:
            # 2~10점 범위를 0~1로 변환
            # (값 - 최소값) / (최대값 - 최소값)
            df[col] = (df[col] - 2) / (10 - 2)
            
            # 혹시라도 범위를 벗어나는 값이 생길 경우를 대비해 0~1 사이로 제한(Clip)
            df[col] = df[col].clip(0, 1)
            
    return df

# 실행
df_traits_scaled = scale_traits_fixed(df_with_traits)

# 확인
print("성향 점수 스케일링 확인 (8점이었던 경우 0.75가 되어야 함):")
print(df_traits_scaled[['parenting_style_SF']].head())

성향 점수 스케일링 확인 (8점이었던 경우 0.75가 되어야 함):
           parenting_style_SF
user_name                    
강현준                     0.500
임경빈                     0.625
이정미                     0.875
쳌                       0.625
김용희                     0.375


## (2) mbti형 생성

In [37]:
scenario_cols = [c for c in df.columns if c.startswith("sc_")]

In [38]:
print("찾은 scenario_ 컬럼 개수:", len(scenario_cols))
scenario_cols

찾은 scenario_ 컬럼 개수: 10


['sc_toothbrushing',
 'sc_bedtime_story',
 'sc_competition_2nd',
 'sc_talent_education',
 'sc_discipline_conflict',
 'sc_play_vs_chores',
 'sc_grandparents_help',
 'sc_inlaws_advice',
 'sc_rainy_zoo',
 'sc_education_fund_risk']

In [39]:
# 1) "두 개씩" 묶기: (0,1), (2,3), (4,5), (6,7), (8,9)
pairs = list(zip(scenario_cols[0::2], scenario_cols[1::2]))

In [40]:
# 2) 5개 축 정의 (왼쪽 글자, 오른쪽 글자, 축 이름 key)
#    - key는 새로 만들 컬럼명에 사용
axes = [
    ("S", "F", "parenting_style_SF"),  # 양육 실행 스타일
    ("A", "H", "education_priority_AH"),  # 교육/성장 우선순위
    ("E", "L", "co_parenting_mode_EL"),  # 공동양육 운영 방식
    ("B", "T", "family_boundary_BT"),  # 확장가족/경계
    ("P", "G", "planning_risk_PG"),  # 계획/리스크 대응
]

In [41]:
# 축 개수와 pair 개수가 맞는지 확인
if len(pairs) != len(axes):
    raise ValueError(f"pairs({len(pairs)})와 axes({len(axes)}) 개수가 맞지 않습니다.")

In [42]:
# 3) 타이브레이크 함수: "무조건 한 쪽 글자"를 반환하는 함수
# v1, v2: 1~5 응답값(결측이면 이미 3으로 채운 뒤 들어오게 처리)
# left/right: 축의 왼쪽/오른쪽 글자
# - signed 변환: (값 - 3)
# - 합계가 양수면 right, 음수면 left
# - 합계가 0이면(중립 가능) 타이브레이크로 중립 제거

def pick_letter_no_neutral(v1: float, v2: float, left: str, right: str) -> str:

    s1 = v1 - 3
    s2 = v2 - 3
    total = s1 + s2

    if total > 0:
        return right
    if total < 0:
        return left

    # ---- 여기부터가 "중립 방지" 타이브레이크 ----
    # 1) 두 번째 문항이 3이 아니면, 그 문항 방향으로 결정
    if s2 > 0:
        return right
    if s2 < 0:
        return left

    # 2) 첫 번째 문항이 3이 아니면, 그 문항 방향으로 결정
    if s1 > 0:
        return right
    if s1 < 0:
        return left

    # 3) 둘 다 정확히 3이면 완전 중립이므로 기본값으로 한쪽 고정
    #    기본값은 오른쪽으로 두었습니다(원하면 왼쪽으로 바꿔도 됨).
    return right

In [43]:
childcare_type = pd.Series("", index=df.index, dtype="string")  # 누적해서 5글자 만들기

for (left, right, axis_key), (col1, col2) in zip(axes, pairs):

    # 4-1) 숫자 변환
    v1 = pd.to_numeric(df[col1], errors="coerce").fillna(3)
    v2 = pd.to_numeric(df[col2], errors="coerce").fillna(3)

    # 4-2) 해당 축의 글자만 계산
    letters = [
        pick_letter_no_neutral(a, b, left, right)
        for a, b in zip(v1.tolist(), v2.tolist())
    ]

    # 4-3) 최종 5글자 문자열에 누적
    childcare_type = childcare_type + pd.Series(letters, index=df.index, dtype="string")

    # 4-4) 로그
    print(f"\n[{axis_key}] {left} vs {right}")
    print(" - 사용 문항 1:", col1)
    print(" - 사용 문항 2:", col2)

# 4-5) 최종 결과 컬럼만 저장
df_traits_scaled["childcare_type_5axis"] = childcare_type

# 확인
print(df_traits_scaled[["childcare_type_5axis"]].head())


[parenting_style_SF] S vs F
 - 사용 문항 1: sc_toothbrushing
 - 사용 문항 2: sc_bedtime_story

[education_priority_AH] A vs H
 - 사용 문항 1: sc_competition_2nd
 - 사용 문항 2: sc_talent_education

[co_parenting_mode_EL] E vs L
 - 사용 문항 1: sc_discipline_conflict
 - 사용 문항 2: sc_play_vs_chores

[family_boundary_BT] B vs T
 - 사용 문항 1: sc_grandparents_help
 - 사용 문항 2: sc_inlaws_advice

[planning_risk_PG] P vs G
 - 사용 문항 1: sc_rainy_zoo
 - 사용 문항 2: sc_education_fund_risk
          childcare_type_5axis
user_name                     
강현준                      FHLTG
임경빈                      FALTG
이정미                      FHLBP
쳌                        FHLBG
김용희                      SAETP


### mbti형 벡터화

In [44]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize

# =========================
# 0) 설정
# =========================
COL_5AXIS = "childcare_type_5axis"   # 5글자 컬럼명
axis_names = ["SF", "AH", "EL", "BT", "PG"]

# 각 축의 가능한 글자(고정) - 순서도 고정됨
axis_categories = [
    ["S", "F"],  # SF
    ["A", "H"],  # AH
    ["E", "L"],  # EL
    ["B", "T"],  # BT
    ["P", "G"],  # PG
]

# =========================
# 1) 5글자 문자열 -> 축별 1글자 컬럼 5개로 분해
# =========================
df_traits_scaled[COL_5AXIS] = df_traits_scaled[COL_5AXIS].astype(str).str.upper().str.strip()

# 길이 5 아닌 값이 있으면 미리 체크(원하면 주석 해제)
bad = df_traits_scaled[COL_5AXIS].str.len() != 5
if bad.any():
    raise ValueError(
        f"{COL_5AXIS}에 길이가 5가 아닌 값이 있습니다. 예: {df.loc[bad, COL_5AXIS].head().tolist()}"
    )

letter_cols = []
for i, ax in enumerate(axis_names):
    c = f"axis_{ax}_letter_from5"     # 임시 분해 컬럼
    df_traits_scaled[c] = df_traits_scaled[COL_5AXIS].str.slice(i, i+1)
    letter_cols.append(c)

# =========================
# 2) 벡터화(One-Hot) - 카테고리 고정
# =========================
# sklearn 버전 호환(sparse_output vs sparse)
try:
    ohe = OneHotEncoder(categories=axis_categories, handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(categories=axis_categories, handle_unknown="ignore", sparse=False)

X_ohe = ohe.fit_transform(df_traits_scaled[letter_cols])  # (n, 10) 0/1

# 컬럼명 얻기(버전 호환)
try:
    feat_names = ohe.get_feature_names_out(letter_cols)
except AttributeError:
    feat_names = ohe.get_feature_names(letter_cols)

# =========================
# 3) 스케일링 + L2 정규화(코사인 유사도용 벡터)
# =========================
# one-hot은 0/1이지만, "스케일링까지" 요청이라 표준화 적용
scaler = StandardScaler(with_mean=False)
X_scaled = scaler.fit_transform(X_ohe)

# 코사인 유사도에 바로 쓰기 좋게 L2 normalize
X_5axis = normalize(X_scaled, norm="l2")  # (n, 10)

# =========================
# 4) (선택) df에 벡터 컬럼 붙이기
# =========================
df_5axis_vec = pd.DataFrame(X_5axis, columns=feat_names, index=df.index)
df_traits_scaled = pd.concat([df_traits_scaled, df_5axis_vec], axis=1)

# 확인
print("벡터 shape:", X_5axis.shape)
print(df_traits_scaled[[COL_5AXIS] + list(df_5axis_vec.columns[:6])].head())

벡터 shape: (34, 10)
          childcare_type_5axis  axis_SF_letter_from5_S  \
user_name                                                
강현준                      FHLTG                0.000000   
임경빈                      FALTG                0.000000   
이정미                      FHLBP                0.000000   
쳌                        FHLBG                0.000000   
김용희                      SAETP                0.490406   

           axis_SF_letter_from5_F  axis_AH_letter_from5_A  \
user_name                                                   
강현준                      0.490406                0.000000   
임경빈                      0.490406                0.520154   
이정미                      0.490406                0.000000   
쳌                        0.490406                0.000000   
김용희                      0.000000                0.520154   

           axis_AH_letter_from5_H  axis_EL_letter_from5_E  \
user_name                                                   
강현준                     

In [45]:
df_traits_scaled.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 강현준 to 유현희
Data columns (total 71 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   ideal_type_강아지상                                   34 non-null     bool   
 1   ideal_type_고양이상                                   34 non-null     bool   
 2   ideal_type_사슴상                                    34 non-null     bool   
 3   ideal_type_토끼상                                    34 non-null     bool   
 4   ideal_type_꼬부기상                                   34 non-null     bool   
 5   ideal_type_곰상                                     34 non-null     bool   
 6   ideal_type_쿼카상                                    34 non-null     bool   
 7   ideal_type_쥐상                                     34 non-null     bool   
 8   ideal_type_도룡뇽상                                   34 non-null     bool   
 9   ideal_type_여우상                      

In [46]:
df_traits_scaled.head()

,ideal_type_강아지상,ideal_type_고양이상,ideal_type_사슴상,ideal_type_토끼상,ideal_type_꼬부기상,ideal_type_곰상,ideal_type_쿼카상,ideal_type_쥐상,ideal_type_도룡뇽상,ideal_type_여우상,...,axis_SF_letter_from5_S,axis_SF_letter_from5_F,axis_AH_letter_from5_A,axis_AH_letter_from5_H,axis_EL_letter_from5_E,axis_EL_letter_from5_L,axis_BT_letter_from5_B,axis_BT_letter_from5_T,axis_PG_letter_from5_P,axis_PG_letter_from5_G
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,False,False,False,False,False,False,False,False,False,False,...,0.000000,0.490406,0.000000,0.520154,0.000000,0.414939,0.000000,0.396586,0.000000,0.399359
임경빈,False,False,False,False,True,False,False,False,False,False,...,0.000000,0.490406,0.520154,0.000000,0.000000,0.414939,0.000000,0.396586,0.000000,0.399359
이정미,False,False,False,False,False,True,False,False,False,False,...,0.000000,0.490406,0.000000,0.520154,0.000000,0.414939,0.396586,0.000000,0.399359,0.000000
쳌,False,True,False,False,False,False,False,False,False,False,...,0.000000,0.490406,0.000000,0.520154,0.000000,0.414939,0.396586,0.000000,0.000000,0.399359
김용희,False,False,False,False,False,False,False,False,False,True,...,0.490406,0.000000,0.520154,0.000000,0.414939,0.000000,0.000000,0.396586,0.399359,0.000000


In [49]:
df_traits_scaled.to_csv('./data/설문조사_데이터_7차.csv')

# 유사도 확인

In [50]:
df = pd.read_csv('./data/설문조사_데이터_7차.csv')

In [51]:
df.head()

,user_name,ideal_type_강아지상,ideal_type_고양이상,ideal_type_사슴상,ideal_type_토끼상,ideal_type_꼬부기상,ideal_type_곰상,ideal_type_쿼카상,ideal_type_쥐상,ideal_type_도룡뇽상,...,axis_SF_letter_from5_S,axis_SF_letter_from5_F,axis_AH_letter_from5_A,axis_AH_letter_from5_H,axis_EL_letter_from5_E,axis_EL_letter_from5_L,axis_BT_letter_from5_B,axis_BT_letter_from5_T,axis_PG_letter_from5_P,axis_PG_letter_from5_G
0,강현준,False,False,False,False,False,False,False,False,False,...,0.000000,0.490406,0.000000,0.520154,0.000000,0.414939,0.000000,0.396586,0.000000,0.399359
1,임경빈,False,False,False,False,True,False,False,False,False,...,0.000000,0.490406,0.520154,0.000000,0.000000,0.414939,0.000000,0.396586,0.000000,0.399359
2,이정미,False,False,False,False,False,True,False,False,False,...,0.000000,0.490406,0.000000,0.520154,0.000000,0.414939,0.396586,0.000000,0.399359,0.000000
3,쳌,False,True,False,False,False,False,False,False,False,...,0.000000,0.490406,0.000000,0.520154,0.000000,0.414939,0.396586,0.000000,0.000000,0.399359
4,김용희,False,False,False,False,False,False,False,False,False,...,0.490406,0.000000,0.520154,0.000000,0.414939,0.000000,0.000000,0.396586,0.399359,0.000000


In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 72 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   user_name                                         34 non-null     str    
 1   ideal_type_강아지상                                   34 non-null     bool   
 2   ideal_type_고양이상                                   34 non-null     bool   
 3   ideal_type_사슴상                                    34 non-null     bool   
 4   ideal_type_토끼상                                    34 non-null     bool   
 5   ideal_type_꼬부기상                                   34 non-null     bool   
 6   ideal_type_곰상                                     34 non-null     bool   
 7   ideal_type_쿼카상                                    34 non-null     bool   
 8   ideal_type_쥐상                                     34 non-null     bool   
 9   ideal_type_도룡뇽상                   

In [53]:
df.set_index('user_name', inplace=True)
df.head(2)

,ideal_type_강아지상,ideal_type_고양이상,ideal_type_사슴상,ideal_type_토끼상,ideal_type_꼬부기상,ideal_type_곰상,ideal_type_쿼카상,ideal_type_쥐상,ideal_type_도룡뇽상,ideal_type_여우상,...,axis_SF_letter_from5_S,axis_SF_letter_from5_F,axis_AH_letter_from5_A,axis_AH_letter_from5_H,axis_EL_letter_from5_E,axis_EL_letter_from5_L,axis_BT_letter_from5_B,axis_BT_letter_from5_T,axis_PG_letter_from5_P,axis_PG_letter_from5_G
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,False,False,False,False,False,False,False,False,False,False,...,0.0,0.490406,0.000000,0.520154,0.0,0.414939,0.0,0.396586,0.0,0.399359
임경빈,False,False,False,False,True,False,False,False,False,False,...,0.0,0.490406,0.520154,0.000000,0.0,0.414939,0.0,0.396586,0.0,0.399359


In [54]:
ideltype_cols = [c for c in df.columns if c.startswith('ideal_')]

df = df.drop(columns=ideltype_cols)
df.head(2)

,p_children_count_1명,p_children_count_2명,p_children_count_3명,p_children_count_그 이상,p_children_count_기타,p_children_composition_오직 딸,p_children_composition_오직 아들,"p_children_composition_딸 1명, 아들 1명",p_children_composition_기타,p_children_timing_결혼 즉시,...,axis_SF_letter_from5_S,axis_SF_letter_from5_F,axis_AH_letter_from5_A,axis_AH_letter_from5_H,axis_EL_letter_from5_E,axis_EL_letter_from5_L,axis_BT_letter_from5_B,axis_BT_letter_from5_T,axis_PG_letter_from5_P,axis_PG_letter_from5_G
user_name,,,,,,,,,,,,,,,,,,,,,
강현준,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.490406,0.000000,0.520154,0.0,0.414939,0.0,0.396586,0.0,0.399359
임경빈,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,...,0.0,0.490406,0.520154,0.000000,0.0,0.414939,0.0,0.396586,0.0,0.399359


In [55]:
print(list([c for c in df.columns if any(q in c for q in ['p_', 'e_', 'child_'])]))

['p_children_count_1명', 'p_children_count_2명', 'p_children_count_3명', 'p_children_count_그 이상', 'p_children_count_기타', 'p_children_composition_오직 딸', 'p_children_composition_오직 아들', 'p_children_composition_딸 1명, 아들 1명', 'p_children_composition_기타', 'p_children_timing_결혼 즉시', 'p_children_timing_결혼 후 1~2년 이내', 'p_children_timing_결혼 후 3~5년 이내', 'p_children_timing_경제적 안정 후', 'p_children_timing_기타', 'p_infertility_alternative_의학적 도움 적극 시도', 'p_infertility_alternative_입양 고려', 'p_infertility_alternative_무자녀', 'p_infertility_alternative_기타', 'e_childcare_cost_share_노후보단 자녀 교육', 'e_childcare_cost_share_노후 먼저, 남는 예산으로 지원', 'e_childcare_cost_share_기타', 'e_parental_leave_burden_경제력 높은 사람 일하고, 한명은 전담 육아', 'e_parental_leave_burden_맞벌이하면서 외부 도움(조부모, 시터)', 'e_parental_leave_burden_기타', 'child_values_open_경제적 성공, 사회적 지위', 'child_values_open_도덕적, 타인 배려', 'child_values_open_자신이 좋아하는 일, 행복', 'child_values_open_회복탄력성, 생활력 강한 사람', 'child_values_open_기타', 'parenting_style_SF', 'co_parenting_mode_EL', 'childca

In [56]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

prefixes = ('p_', 'e_', 'child_')
val_cols = [c for c in df.columns if c.startswith(prefixes)]
trait_cols = ['parenting_style_SF', 'education_priority_AH', 'co_parenting_mode_EL', 'family_boundary_BT', 'planning_risk_PG']

#유사도 계산
val_sim_matrix = cosine_similarity(df[val_cols], df[val_cols])
val_sim_matrix

array([[1.        , 0.59071735, 0.36380344, ..., 0.61579354, 0.22140372,
        0.12249899],
       [0.59071735, 1.        , 0.53425846, ..., 0.83992105, 0.22951012,
        0.42857143],
       [0.36380344, 0.53425846, 1.        , ..., 0.37416574, 0.10954451,
        0.64649763],
       ...,
       [0.61579354, 0.83992105, 0.37416574, ..., 1.        , 0.58554004,
        0.2159797 ],
       [0.22140372, 0.22951012, 0.10954451, ..., 0.58554004, 1.        ,
        0.07377111],
       [0.12249899, 0.42857143, 0.64649763, ..., 0.2159797 , 0.07377111,
        1.        ]], shape=(34, 34))

In [57]:
val_sim_df = pd.DataFrame(
    val_sim_matrix,
    index = df.index,
    columns= df.index
)

In [59]:
#가치관 점수는 유사하고, 사니라오 점수는 유사하거나 반대되는 사람 찾기
def get_smart_recommendations(user_name, top_n=5):
    if user_name not in df.index:
        return '사용자를 찾을 수 없습니다.'
    
    #내 MBTI 가져오기
    my_traits = df.loc[user_name, trait_cols]

    #모든 유저 리스트 생성
    recommendations = []
    for other_user in df.index:
        if user_name == other_user:
            continue

        #가치관 유사도 점수
        val_score = val_sim_df.loc[user_name, other_user]

        #시나리오 성향 계산 
        other_traits = df.loc[other_user, trait_cols]
        trait_diff = np.abs(my_traits - other_traits).mean()

        #유저별 가치관 유사도, 시나리오 성향 차이 리스트로 정리
        recommendations.append({
            'name' : other_user,
            'val_score' : val_score,
            'trait_diff' : trait_diff,
            'trait_type' : '보완형(Flipped)' if trait_diff > 0.5 else '동질형(Similar)'
        })
    
    recom_df = pd.DataFrame(recommendations).sort_values(by='val_score', ascending=False)